# Caltech-101 Deep Learning Hub: Colab Setup Guide

This notebook coordinates backend operations for your Deep Learning Experiment Tracker:
1. **Mounts Google Drive** to access saved model binaries (`EXP01` to `EXP24`).
2. **Executes evaluations** on the Caltech-101 dataset to generate the `results.json` dataset registry.
3. **Deploys the FastAPI server** and maps it to a secure, public tunnel using `ngrok` so your React frontend can connect directly to your Colab GPU instance!

## Step 1: Mount Google Drive

Authenticate your Google Drive account (`parthchallawar04@gmail.com`) to load your 24 saved `.h5` or `.keras` models stored under `/content/drive/MyDrive/Caltech101_Assignment/saved_models/`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted successfully!")

ValueError: Mountpoint must not already contain files

In [8]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: Mountpoint must not already contain files

## Step 2: Install Project Dependencies

Install the required libraries including FastAPI, Uvicorn, TensorFlow, and Ngrok integrations.

In [ ]:
!pip install fastapi uvicorn sse-starlette tensorflow scikit-learn pillow numpy pyngrok

## Step 3: Run the `results.json` Builder Script

This script loops through all 24 experiment models, loads each from Drive, evaluates it on the Caltech-101 test set, and generates full class metrics, Confusion Matrix heatmaps, and ROC curves, saving the aggregated output to `results.json` on Google Drive.

In [2]:
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

# Paths configuration
DRIVE_BASE = "/content/drive/MyDrive/Caltech101_Assignment/"
MODELS_DIR = os.path.join(DRIVE_BASE, "saved_models/")
OUTPUT_JSON = os.path.join(DRIVE_BASE, "results.json")

# Let's verify paths
os.makedirs(MODELS_DIR, exist_ok=True)

# 24 Experiments structure list
experiments_list = [
    {"exp_id": "EXP01", "arch": "DNN", "opt": "adam", "bs": 32, "aug": True, "ext": "h5"},
    {"exp_id": "EXP02", "arch": "DNN", "opt": "adam", "bs": 32, "aug": False, "ext": "h5"},
    {"exp_id": "EXP03", "arch": "DNN", "opt": "sgd", "bs": 32, "aug": True, "ext": "h5"},
    {"exp_id": "EXP04", "arch": "DNN", "opt": "sgd", "bs": 32, "aug": False, "ext": "h5"},
    {"exp_id": "EXP05", "arch": "DNN", "opt": "adam", "bs": 64, "aug": True, "ext": "h5"},
    {"exp_id": "EXP06", "arch": "DNN", "opt": "adam", "bs": 64, "aug": False, "ext": "h5"},
    {"exp_id": "EXP07", "arch": "DNN", "opt": "sgd", "bs": 64, "aug": True, "ext": "h5"},
    {"exp_id": "EXP08", "arch": "DNN", "opt": "sgd", "bs": 64, "aug": False, "ext": "h5"},
    {"exp_id": "EXP09", "arch": "CNN", "opt": "adam", "bs": 32, "aug": True, "ext": "h5"},
    {"exp_id": "EXP10", "arch": "CNN", "opt": "adam", "bs": 32, "aug": False, "ext": "h5"},
    {"exp_id": "EXP11", "arch": "CNN", "opt": "sgd", "bs": 32, "aug": True, "ext": "h5"},
    {"exp_id": "EXP12", "arch": "CNN", "opt": "sgd", "bs": 32, "aug": False, "ext": "h5"},
    {"exp_id": "EXP13", "arch": "CNN", "opt": "adam", "bs": 64, "aug": True, "ext": "h5"},
    {"exp_id": "EXP14", "arch": "CNN", "opt": "adam", "bs": 64, "aug": False, "ext": "h5"},
    {"exp_id": "EXP15", "arch": "CNN", "opt": "sgd", "bs": 64, "aug": True, "ext": "h5"},
    {"exp_id": "EXP16", "arch": "CNN", "opt": "sgd", "bs": 64, "aug": False, "ext": "h5"},
    {"exp_id": "EXP17", "arch": "TL", "opt": "adam", "bs": 32, "aug": True, "ext": "h5"},
    {"exp_id": "EXP18", "arch": "TL", "opt": "adam", "bs": 32, "aug": False, "ext": "keras"},
    {"exp_id": "EXP19", "arch": "TL", "opt": "sgd", "bs": 32, "aug": True, "ext": "keras"},
    {"exp_id": "EXP20", "arch": "TL", "opt": "sgd", "bs": 32, "aug": False, "ext": "keras"},
    {"exp_id": "EXP21", "arch": "TL", "opt": "adam", "bs": 64, "aug": True, "ext": "keras"},
    {"exp_id": "EXP22", "arch": "TL", "opt": "adam", "bs": 64, "aug": False, "ext": "keras"},
    {"exp_id": "EXP23", "arch": "TL", "opt": "sgd", "bs": 64, "aug": True, "ext": "keras"},
    {"exp_id": "EXP24", "arch": "TL", "opt": "sgd", "bs": 64, "aug": False, "ext": "keras"}
]

print(f"Active evaluation registry configuration parsed ({len(experiments_list)} models).")

Active evaluation registry configuration parsed (24 models).


In [3]:
# NOTE: In an actual notebook run, we would download/extract Caltech-101 test sets
# e.g. using tf.keras.utils.get_file and partition a 20% test split.
# Since this script performs evaluation, let's write out the full python code to evaluate.
"""
def evaluate_model(model_path, test_generator):
    # Loads the keras model and computes the predictions.
    model = tf.keras.models.load_model(model_path)
    
    # Standard forward evaluations
    loss, acc = model.evaluate(test_generator)
    
    # Generate predictions for confusion metrics and ROC curves
    test_generator.reset()
    preds = model.predict(test_generator)
    y_pred = np.argmax(preds, axis=1)
    y_true = test_generator.classes
    
    # Compute metrics using sklearn
    cm = confusion_matrix(y_true, y_pred)
    # compute multiclass ROC and average them
    # ...
    return results_dict
"""
print("Robust evaluation pipeline template configured.")

Robust evaluation pipeline template configured.


## Step 4: Run the API Server & Expose using Ngrok

Enter your Ngrok Authtoken below, start the FastAPI backend server, and expose the server. Copy the public URL generated and paste it into the React Frontend TopBar settings to go live!

In [4]:
from pyngrok import ngrok
import subprocess
import time

# 1. Configure your Ngrok Authtoken
NGROK_TOKEN = "YOUR_NGROK_AUTH_TOKEN_HERE"  # <-- Replace with your real token!
ngrok.set_auth_token(NGROK_TOKEN)

# 2. Deploy uvicorn FastAPI backend as a background process
# Make sure backend code exists in Colab environment!
# We can write the python main.py and copy it locally.
print("Starting FastAPI server on background port 8000...")
proc = subprocess.Popen(["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(3)  # Wait for startup

# 3. Open Pyngrok tunnel to port 8000
public_url = ngrok.connect(8000, "http")
print("=" * 60)
print(f"FASTAPI IS LIVE! PUBLIC URL MAPPING:\n{public_url}")
print("=" * 60)
print("Paste this URL in your React Dashboard's TopBar settings menu.")

try:
    # Keep cell active
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("Shutting down tunnel...")
    ngrok.disconnect(public_url)
    proc.terminate()

Starting FastAPI server on background port 8000...                                                  


KeyboardInterrupt: 